# VAW Phase 1 — Colab run: M8b (Algorithm 1 interaction scores) + verification
Run notebooks 01 (M1/M2) and 02 (M8a) first. M8b needs **no GPU** and takes seconds, because it reads the cached tracks.
Full explanation of every output and how to check it: `docs/VERIFY_M8.md`.

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/KRITHIKSAI16/Violence-Against-Women.git"
if not os.path.isdir('/content/VAW'):
    !git clone -q {url} /content/VAW
else:
    !git -C /content/VAW pull -q {url} main
%cd /content/VAW
!pip install -q -r requirements.txt
!python -m pytest -q

## 1. Run M8b on every tracked clip
Prints one line per clip: frames, pair-frames (frames x pairs scored), mean / max / 95th-percentile score. `(no interaction)` = fewer than 2 people were ever tracked together.

In [ ]:
!python -m src.assm.interaction --config configs/colab.yaml

## 2. Per-category summary (the first look at buildup by category)

In [ ]:
import glob, csv, numpy as np, collections
S = '/content/drive/MyDrive/VAW_results/scores'
by = collections.defaultdict(list)
for f in sorted(glob.glob(f'{S}/*/*_curve.csv')):
    cat = f.split('/')[-2]; cid = f.split('/')[-1].replace('_curve.csv', '')
    sc = np.array([float(r['max_score']) for r in csv.DictReader(open(f))])
    by[cat].append((cid, sc))
print(f"{'category':<16}{'clips':>6}{'no-interaction':>16}{'mean of means':>15}{'mean of p95':>13}")
for cat, lst in sorted(by.items()):
    none = sum(1 for _, s in lst if s.max() == 0)
    act = [s for _, s in lst if s.max() > 0]
    print(f"{cat:<16}{len(lst):>6}{none:>16}{np.mean([s.mean() for s in act]) if act else 0:>15.2f}{np.mean([np.percentile(s,95) for s in act]) if act else 0:>13.2f}")

## 3. Verify one clip by eye: annotated video
Change `CLIP` to any clip id. Red line = top pair above `--tau`; the strip at the bottom is the buildup curve with a moving cursor.

In [ ]:
CLIP = 'Stalking_v3'
!python -m src.assm.render {CLIP} --config configs/colab.yaml --tau 3.0
from IPython.display import Video
Video(f'/content/drive/MyDrive/VAW_results/scores/videos/{CLIP}.mp4', embed=True, width=480)

## 4. Verify the numbers: recompute the top-scoring row by hand
Every line should say `OK`. Use `--frame N` to check a specific frame (the frame number is shown in the video's top bar as t = frame / 30).

In [ ]:
!python -m src.assm.hand_check {CLIP} --config configs/colab.yaml